# Notebook 04 — Modelo Analítico + ML
## NYC Taxi Trips — Big Data Pipeline

**Objetivo:** Crear el modelo analítico (star schema) en PostgreSQL y aplicar modelos de ML.

**Componentes:**
- Star schema: fact_trips + dimensiones
- Clustering KMeans de zonas de riesgo/demanda
- Modelo de predicción de tarifa (Random Forest)
- Guardar resultados en capa curated

In [1]:
# ─── Celda 1: Imports ─────────────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
import psycopg2
from psycopg2.extras import execute_values
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

BASE_PATH      = os.path.abspath("../")
PROCESSED_PATH = os.path.join(BASE_PATH, "data", "processed")
CURATED_PATH   = os.path.join(BASE_PATH, "data", "curated")
SCREENS_PATH   = os.path.join(BASE_PATH, "screenshots")

os.makedirs(CURATED_PATH, exist_ok=True)

print("Imports completados ✅")

Imports completados ✅


In [2]:
# ─── Celda 2: Cargar datos procesados ────────────────────────────────────────
print("Cargando datos procesados...")

pdf = pd.read_parquet(os.path.join(PROCESSED_PATH, "trips_processed.parquet"))

print(f"Filas cargadas: {len(pdf):,}")
print(f"Columnas      : {list(pdf.columns)}")
print("Carga completada ✅")

Cargando datos procesados...
Filas cargadas: 8,780,868
Columnas      : ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'tip_amount', 'total_amount', 'pickup_hour', 'pickup_day', 'pickup_month', 'pickup_weekday', 'is_weekend', 'trip_duration_min', 'avg_speed_mph', 'cost_per_mile', 'time_of_day', 'payment_type_name', 'pickup_zone', 'pickup_borough', 'dropoff_zone', 'dropoff_borough']
Carga completada ✅


In [3]:
# ─── Celda 3: Star Schema — Dimensiones ──────────────────────────────────────
print("Creando dimensiones del Star Schema...")

# dim_date
pdf["pickup_dt"] = pd.to_datetime(pdf["tpep_pickup_datetime"])
dim_date = pdf[["pickup_dt"]].copy()
dim_date["date"]       = dim_date["pickup_dt"].dt.date
dim_date["year"]       = dim_date["pickup_dt"].dt.year
dim_date["month"]      = dim_date["pickup_dt"].dt.month
dim_date["day"]        = dim_date["pickup_dt"].dt.day
dim_date["hour"]       = dim_date["pickup_dt"].dt.hour
dim_date["weekday"]    = dim_date["pickup_dt"].dt.weekday
dim_date["day_name"]   = dim_date["pickup_dt"].dt.day_name()
dim_date["is_weekend"] = (dim_date["weekday"] >= 5).astype(int)
dim_date["time_of_day"] = pdf["time_of_day"]
dim_date = dim_date.drop(columns=["pickup_dt"]).drop_duplicates().reset_index(drop=True)
dim_date["date_id"] = dim_date.index + 1
print(f"dim_date      : {len(dim_date):,} filas")

# dim_location
pickup_locs = pdf[["PULocationID","pickup_zone","pickup_borough"]].rename(
    columns={"PULocationID":"location_id","pickup_zone":"zone","pickup_borough":"borough"})
dropoff_locs = pdf[["DOLocationID","dropoff_zone","dropoff_borough"]].rename(
    columns={"DOLocationID":"location_id","dropoff_zone":"zone","dropoff_borough":"borough"})
dim_location = pd.concat([pickup_locs, dropoff_locs]).drop_duplicates(
    subset=["location_id"]).dropna().reset_index(drop=True)
print(f"dim_location  : {len(dim_location):,} filas")

# dim_payment
dim_payment = pdf[["payment_type","payment_type_name"]].drop_duplicates().reset_index(drop=True)
dim_payment["payment_id"] = dim_payment.index + 1
print(f"dim_payment   : {len(dim_payment):,} filas")

# fact_trips (muestra de 500k para star schema)
fact_trips = pdf[[
    "VendorID","PULocationID","DOLocationID","payment_type",
    "pickup_hour","pickup_month","is_weekend",
    "passenger_count","trip_distance","fare_amount",
    "tip_amount","total_amount","trip_duration_min",
    "avg_speed_mph","cost_per_mile","time_of_day",
    "pickup_borough","dropoff_borough"
]].head(500000).reset_index(drop=True)
fact_trips["trip_id"] = fact_trips.index + 1
print(f"fact_trips    : {len(fact_trips):,} filas")

print("\nStar Schema creado ✅")

Creando dimensiones del Star Schema...
dim_date      : 2,201 filas
dim_location  : 258 filas
dim_payment   : 4 filas
fact_trips    : 500,000 filas

Star Schema creado ✅


In [4]:
# ─── Celda 4: Cargar Star Schema a PostgreSQL ─────────────────────────────────
print("Cargando Star Schema a PostgreSQL...")

conn = psycopg2.connect(
    host="localhost", port=5432,
    database="nyc_taxi_db", user="postgres", password=""
)
cur = conn.cursor()

# dim_location
cur.execute("""
DROP TABLE IF EXISTS dim_location CASCADE;
CREATE TABLE dim_location (
    location_id INT PRIMARY KEY,
    zone        TEXT,
    borough     TEXT
);
""")
execute_values(cur, "INSERT INTO dim_location VALUES %s",
    [tuple(r) for r in dim_location[["location_id","zone","borough"]].itertuples(index=False)])
print("  dim_location cargada ✅")

# dim_payment
cur.execute("""
DROP TABLE IF EXISTS dim_payment CASCADE;
CREATE TABLE dim_payment (
    payment_id   INT PRIMARY KEY,
    payment_type INT,
    payment_name TEXT
);
""")
execute_values(cur, "INSERT INTO dim_payment VALUES %s",
    [tuple(r) for r in dim_payment[["payment_id","payment_type","payment_type_name"]].itertuples(index=False)])
print("  dim_payment cargada ✅")

# fact_trips
cur.execute("""
DROP TABLE IF EXISTS fact_trips CASCADE;
CREATE TABLE fact_trips (
    trip_id          INT PRIMARY KEY,
    vendor_id        INT,
    pu_location_id   INT,
    do_location_id   INT,
    payment_type     INT,
    pickup_hour      INT,
    pickup_month     INT,
    is_weekend       INT,
    passenger_count  FLOAT,
    trip_distance    FLOAT,
    fare_amount      FLOAT,
    tip_amount       FLOAT,
    total_amount     FLOAT,
    trip_duration_min FLOAT,
    avg_speed_mph    FLOAT,
    cost_per_mile    FLOAT,
    time_of_day      TEXT,
    pickup_borough   TEXT,
    dropoff_borough  TEXT
);
""")

# Insertar en chunks de 10k
chunk_size = 10000
cols = ["trip_id","VendorID","PULocationID","DOLocationID","payment_type",
        "pickup_hour","pickup_month","is_weekend","passenger_count",
        "trip_distance","fare_amount","tip_amount","total_amount",
        "trip_duration_min","avg_speed_mph","cost_per_mile",
        "time_of_day","pickup_borough","dropoff_borough"]

for i in range(0, len(fact_trips), chunk_size):
    chunk = fact_trips[cols].iloc[i:i+chunk_size]
    execute_values(cur,
        "INSERT INTO fact_trips VALUES %s",
        [tuple(r) for r in chunk.itertuples(index=False)],
        page_size=500
    )
    if i % 100000 == 0:
        print(f"  Insertando fact_trips... {i:,}/{len(fact_trips):,}")
    conn.commit()

print("  fact_trips cargada ✅")

# Verificar
for tabla in ["dim_location","dim_payment","fact_trips"]:
    cur.execute(f"SELECT COUNT(*) FROM {tabla}")
    print(f"  {tabla:<20} → {cur.fetchone()[0]:,} filas")

cur.close()
conn.close()
print("\nStar Schema en PostgreSQL ✅")

Cargando Star Schema a PostgreSQL...
  dim_location cargada ✅
  dim_payment cargada ✅
  Insertando fact_trips... 0/500,000
  Insertando fact_trips... 100,000/500,000
  Insertando fact_trips... 200,000/500,000
  Insertando fact_trips... 300,000/500,000
  Insertando fact_trips... 400,000/500,000
  fact_trips cargada ✅
  dim_location         → 258 filas
  dim_payment          → 4 filas
  fact_trips           → 500,000 filas

Star Schema en PostgreSQL ✅


In [5]:
# ─── Celda 5: ML — Clustering KMeans de zonas ────────────────────────────────
print("Aplicando KMeans clustering por zona...")

# Agregar métricas por zona de origen
zone_stats = pdf.groupby("PULocationID").agg(
    total_trips    = ("trip_distance",    "count"),
    avg_fare       = ("fare_amount",      "mean"),
    avg_distance   = ("trip_distance",    "mean"),
    avg_duration   = ("trip_duration_min","mean"),
    avg_tip        = ("tip_amount",       "mean"),
    total_revenue  = ("total_amount",     "sum")
).reset_index()

# Merge con nombres de zonas
zone_stats = zone_stats.merge(
    pdf[["PULocationID","pickup_zone","pickup_borough"]].drop_duplicates(),
    on="PULocationID", how="left"
)

# Features para clustering
features = ["total_trips","avg_fare","avg_distance","avg_duration","avg_tip"]
X = zone_stats[features].fillna(0)

# Normalizar
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# KMeans con 4 clusters
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
zone_stats["cluster"] = kmeans.fit_predict(X_scaled)

# Etiquetar clusters
cluster_means = zone_stats.groupby("cluster")["total_trips"].mean()
cluster_rank  = cluster_means.rank(ascending=False).astype(int)
labels = {k: f"Cluster_{v}" for k, v in cluster_rank.items()}
zone_stats["cluster_label"] = zone_stats["cluster"].map(labels)

print("\n=== Distribución de clusters ===")
print(zone_stats.groupby("cluster_label").agg(
    zonas        = ("PULocationID",  "count"),
    avg_trips    = ("total_trips",   "mean"),
    avg_fare     = ("avg_fare",      "mean"),
    avg_revenue  = ("total_revenue", "mean")
).round(2))

print("\nTop 5 zonas por demanda:")
print(zone_stats.sort_values("total_trips", ascending=False)[
    ["pickup_zone","pickup_borough","total_trips","avg_fare","cluster_label"]
].head(10).to_string(index=False))

print("\nKMeans clustering completado ✅")

Aplicando KMeans clustering por zona...

=== Distribución de clusters ===
               zonas  avg_trips  avg_fare  avg_revenue
cluster_label                                         
Cluster_1         33  212056.55     14.84   4819599.27
Cluster_2         25   30776.12     57.07   2249530.86
Cluster_3        114    8709.62     23.79    230033.29
Cluster_4         89     232.61     37.05      9237.60

Top 5 zonas por demanda:
                 pickup_zone pickup_borough  total_trips  avg_fare cluster_label
                 JFK Airport         Queens       436449 62.438944     Cluster_2
       Upper East Side South      Manhattan       415916 12.543493     Cluster_1
              Midtown Center      Manhattan       411090 15.599722     Cluster_1
       Upper East Side North      Manhattan       381305 13.265092     Cluster_1
                Midtown East      Manhattan       317751 15.235703     Cluster_1
Penn Station/Madison Sq West      Manhattan       315164 16.080705     Cluster_1
   

In [6]:
# ─── Celda 6: ML — Predicción de tarifa con Random Forest ─────────────────────
print("Entrenando modelo de predicción de tarifa...")

# Muestra de 100k para entrenar (más rápido)
sample = pdf.sample(100000, random_state=42).copy()

# Encodear variables categóricas
le_tod  = LabelEncoder()
le_boro = LabelEncoder()
sample["time_of_day_enc"]     = le_tod.fit_transform(sample["time_of_day"].fillna("Unknown"))
sample["pickup_borough_enc"]  = le_boro.fit_transform(sample["pickup_borough"].fillna("Unknown"))

features = [
    "trip_distance", "pickup_hour", "pickup_month", "is_weekend",
    "passenger_count", "PULocationID", "DOLocationID",
    "time_of_day_enc", "pickup_borough_enc", "trip_duration_min"
]

X = sample[features].fillna(0)
y = sample["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")

# Entrenar Random Forest
rf = RandomForestRegressor(
    n_estimators=100, max_depth=10,
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)

# Evaluar
y_pred = rf.predict(X_test)
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print(f"\n=== Resultados del modelo ===")
print(f"MAE  (Error promedio): ${mae:.2f}")
print(f"R²   (Precisión)     : {r2:.4f} ({r2*100:.1f}%)")

# Feature importance
importance_df = pd.DataFrame({
    "feature"   : features,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print("\n=== Importancia de variables ===")
print(importance_df.to_string(index=False))

print("\nRandom Forest entrenado ✅")

Entrenando modelo de predicción de tarifa...
Train: 80,000 | Test: 20,000

=== Resultados del modelo ===
MAE  (Error promedio): $0.80
R²   (Precisión)     : 0.9643 (96.4%)

=== Importancia de variables ===
           feature  importance
     trip_distance    0.931985
 trip_duration_min    0.048786
      DOLocationID    0.011219
      PULocationID    0.002652
       pickup_hour    0.002160
pickup_borough_enc    0.001080
   passenger_count    0.000653
      pickup_month    0.000632
   time_of_day_enc    0.000548
        is_weekend    0.000285

Random Forest entrenado ✅


In [8]:
# ─── Celda 7: Guardar resultados ML en PostgreSQL y curated ──────────────────
print("Guardando resultados ML...")

# Guardar clustering en PostgreSQL
conn = psycopg2.connect(
    host="localhost", port=5432,
    database="nyc_taxi_db", user="postgres", password=""
)
cur = conn.cursor()

cur.execute("""
DROP TABLE IF EXISTS zone_clusters CASCADE;
CREATE TABLE zone_clusters (
    location_id    INT,
    zone           TEXT,
    borough        TEXT,
    total_trips    INT,
    avg_fare       FLOAT,
    avg_distance   FLOAT,
    total_revenue  FLOAT,
    cluster        INT,
    cluster_label  TEXT
);
""")

cols_cluster = ["PULocationID","pickup_zone","pickup_borough",
                "total_trips","avg_fare","avg_distance",
                "total_revenue","cluster","cluster_label"]
execute_values(cur, "INSERT INTO zone_clusters VALUES %s",
    [tuple(r) for r in zone_stats[cols_cluster].fillna("").itertuples(index=False)])
conn.commit()
print("  zone_clusters cargada ✅")

# Guardar métricas del modelo
cur.execute("""
DROP TABLE IF EXISTS ml_model_metrics CASCADE;
CREATE TABLE ml_model_metrics (
    model_name   TEXT,
    metric_name  TEXT,
    metric_value FLOAT,
    trained_at   TIMESTAMP DEFAULT NOW()
);
""")
from psycopg2.extras import execute_values

execute_values(
    cur,
    "INSERT INTO ml_model_metrics (model_name, metric_name, metric_value) VALUES %s",
    [("RandomForest_FarePrediction", "MAE",  mae),
     ("RandomForest_FarePrediction", "R2",   r2),
     ("KMeans_ZoneClustering",       "n_clusters", 4)]
)

cur.close()
conn.close()
print("  ml_model_metrics cargada ✅")

# Guardar en curated
zone_stats.to_parquet(os.path.join(CURATED_PATH, "zone_clusters.parquet"), index=False)
importance_df.to_parquet(os.path.join(CURATED_PATH, "feature_importance.parquet"), index=False)
fact_trips.head(500000).to_parquet(os.path.join(CURATED_PATH, "fact_trips_curated.parquet"), index=False)

print("  Archivos curated guardados ✅")
print("\n✅ Notebook 04 completado exitosamente.")
print(f"   Tablas en PostgreSQL: dim_location, dim_payment, fact_trips, zone_clusters, ml_model_metrics")

Guardando resultados ML...
  zone_clusters cargada ✅
  ml_model_metrics cargada ✅
  Archivos curated guardados ✅

✅ Notebook 04 completado exitosamente.
   Tablas en PostgreSQL: dim_location, dim_payment, fact_trips, zone_clusters, ml_model_metrics
